In [30]:
import pandas as pd
import re
from pathlib import Path

In [31]:
path = Path("../1.raw/ds_salaries.csv")
df1 = pd.read_csv(path)

path = Path("../1.raw/global_tech_salary.txt")
df2 = pd.read_csv(path)

path = Path("../1.raw/salary_data_cleaned.csv")
df3 = pd.read_csv(path)

In [32]:
def quick_peek(df, name: str, n=3):
    print(f"\n===== {name} =====")
    print("shape:", df.shape)
    print("cols:", list(df.columns))
    display(df.head(n))

quick_peek(df1, "df1")
quick_peek(df2, "df2")
quick_peek(df3, "df3")



===== df1 =====
shape: (607, 12)
cols: ['Unnamed: 0', 'work_year', 'experience_level', 'employment_type', 'job_title', 'salary', 'salary_currency', 'salary_in_usd', 'employee_residence', 'remote_ratio', 'company_location', 'company_size']


,Unnamed: 0,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,0,2020,MI,FT,Data Scientist,70000,EUR,79833,DE,0,DE,L
1,1,2020,SE,FT,Machine Learning Scientist,260000,USD,260000,JP,0,JP,S
2,2,2020,SE,FT,Big Data Engineer,85000,GBP,109024,GB,50,GB,M



===== df2 =====
shape: (5000, 11)
cols: ['work_year', 'experience_level', 'employment_type', 'job_title', 'salary', 'salary_currency', 'salary_in_usd', 'employee_residence', 'remote_ratio', 'company_location', 'company_size']


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2023,MI,FT,Data Analyst,165000,USD,165000,AU,0,AU,M
1,2023,MI,FT,Data Analyst,70000,USD,70000,US,100,US,M
2,2024,MI,FT,Machine Learning Engineer,85000,EUR,94444,IE,100,IE,M



===== df3 =====
shape: (742, 28)
cols: ['Job Title', 'Salary Estimate', 'Job Description', 'Rating', 'Company Name', 'Location', 'Headquarters', 'Size', 'Founded', 'Type of ownership', 'Industry', 'Sector', 'Revenue', 'Competitors', 'hourly', 'employer_provided', 'min_salary', 'max_salary', 'avg_salary', 'company_txt', 'job_state', 'same_state', 'age', 'python_yn', 'R_yn', 'spark', 'aws', 'excel']


,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,Type of ownership,...,avg_salary,company_txt,job_state,same_state,age,python_yn,R_yn,spark,aws,excel
0,Data Scientist,$53K-$91K (Glassdoor est.),"Data Scientist\nLocation: Albuquerque, NM\nEdu...",3.8,Tecolote Research\n3.8,"Albuquerque, NM","Goleta, CA",501 to 1000 employees,1973,Company - Private,...,72.0,Tecolote Research\n,NM,0,47,1,0,0,0,1
1,Healthcare Data Scientist,$63K-$112K (Glassdoor est.),What You Will Do:\n\nI. General Summary\n\nThe...,3.4,University of Maryland Medical System\n3.4,"Linthicum, MD","Baltimore, MD",10000+ employees,1984,Other Organization,...,87.5,University of Maryland Medical System\n,MD,0,36,1,0,0,0,0
2,Data Scientist,$80K-$90K (Glassdoor est.),"KnowBe4, Inc. is a high growth information sec...",4.8,KnowBe4\n4.8,"Clearwater, FL","Clearwater, FL",501 to 1000 employees,2010,Company - Private,...,85.0,KnowBe4\n,FL,1,10,1,0,1,0,1


In [33]:
TARGET_KEYS = [
    "job", "title", "role",
    "salary", "comp", "usd",
    "year", "work_year",
    "country", "location", "state", "region",
    "remote",
    "company", "size", "org",
    "employment", "contract", "ft", "pt",
    "experience", "seniority", "level"
]

def find_cols(df, keys):
    cols = list(df.columns)
    hits = {}
    for k in keys:
        hits[k] = [c for c in cols if k.lower() in c.lower()]
    return hits

for name, df in [("df1", df1), ("df2", df2), ("df3", df3)]:
    hits = find_cols(df, TARGET_KEYS)
    print(f"\n--- {name} candidates ---")
    for k, lst in hits.items():
        if lst:
            print(f"{k:>10}: {lst}")



--- df1 candidates ---
       job: ['job_title']
     title: ['job_title']
    salary: ['salary', 'salary_currency', 'salary_in_usd']
      comp: ['company_location', 'company_size']
       usd: ['salary_in_usd']
      year: ['work_year']
 work_year: ['work_year']
  location: ['company_location']
    remote: ['remote_ratio']
   company: ['company_location', 'company_size']
      size: ['company_size']
employment: ['employment_type']
experience: ['experience_level']
     level: ['experience_level']

--- df2 candidates ---
       job: ['job_title']
     title: ['job_title']
    salary: ['salary', 'salary_currency', 'salary_in_usd']
      comp: ['company_location', 'company_size']
       usd: ['salary_in_usd']
      year: ['work_year']
 work_year: ['work_year']
  location: ['company_location']
    remote: ['remote_ratio']
   company: ['company_location', 'company_size']
      size: ['company_size']
employment: ['employment_type']
experience: ['experience_level']
     level: ['experience_

In [34]:
def clean_str(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    s = re.sub(r"\s+", " ", s)
    return s if s else None

def coerce_year(x):
    if pd.isna(x):
        return None
    s = str(x)
    m = re.search(r"(19|20)\d{2}", s)
    return int(m.group(0)) if m else None

def coerce_salary_usd(x):
    """Convierte a float si puede; ignora símbolos, comas, etc."""
    if pd.isna(x):
        return None
    s = str(x).strip()
    # quitar moneda/símbolos y separadores
    s = s.replace(",", "")
    s = re.sub(r"[^\d\.]", "", s)
    if s == "" or s == ".":
        return None
    try:
        val = float(s)
        # filtros básicos
        if val <= 0:
            return None
        return val
    except Exception:
        return None


In [35]:
COMMON_COLS = [
    "source_dataset",
    "work_year",
    "role_raw",
    "role_label",          # de momento lo dejamos como placeholder (luego lo construiremos)
    "seniority_raw",
    "seniority",
    "country",
    "state_or_region",
    "remote_ratio",
    "company_size",
    "employment_type",
    "salary_in_usd",
]

def map_experience_level_to_seniority(exp):
    """
    Mapeo típico del dataset DS Salaries:
    EN=Entry, MI=Mid, SE=Senior, EX=Executive
    """
    if pd.isna(exp):
        return None
    exp = str(exp).strip().upper()
    mapping = {
        "EN": "junior",
        "MI": "mid",
        "SE": "senior",
        "EX": "lead",   # o "executive"; elige uno y sé consistente
    }
    return mapping.get(exp, None)

def normalize_df_ds_salaries(df, source_name: str):
    out = pd.DataFrame()

    out["source_dataset"]  = source_name
    out["work_year"]       = df["work_year"].astype("Int64")
    out["role_raw"]        = df["job_title"].astype(str).str.strip()

    out["role_label"]      = None  # lo crearemos después como role_label_salary

    out["seniority_raw"]   = df["experience_level"].astype(str).str.strip()
    out["seniority"]       = df["experience_level"].apply(map_experience_level_to_seniority)

    # En estos datasets ya viene en ISO2
    out["country"]         = df["employee_residence"].astype(str).str.strip()

    out["state_or_region"] = None  # no existe aquí

    # suele ser 0/50/100
    out["remote_ratio"]    = pd.to_numeric(df["remote_ratio"], errors="coerce")

    out["company_size"]    = df["company_size"].astype(str).str.strip()
    out["employment_type"] = df["employment_type"].astype(str).str.strip()

    out["salary_in_usd"]   = pd.to_numeric(df["salary_in_usd"], errors="coerce")

    return out[COMMON_COLS]

def normalize_df_glassdoor(df, source_name="glassdoor"):
    """
    df3 no trae: work_year, employment_type, remote_ratio, country en ISO2.
    Vamos a:
      - salary_in_usd: usar avg_salary (en miles de $) -> * 1000
      - country: asumir US (Glassdoor US típico) -> 'US' (si luego detectas lo contrario, lo cambiamos)
      - state_or_region: job_state
      - company_size: Size
      - employment_type: Unknown
      - remote_ratio: NaN
      - seniority: derivar de Job Title (reglas simples)
    """
    out = pd.DataFrame()

    out["source_dataset"]  = source_name

    # work_year: no existe en df3 -> NaN por ahora (luego decidimos si dropear estas filas o imputar)
    out["work_year"]       = pd.Series([pd.NA] * len(df), dtype="Int64")

    out["role_raw"]        = df["Job Title"].astype(str).str.strip()
    out["role_label"]      = None  # lo crearemos después

    # seniority_raw: el propio título (para debug)
    out["seniority_raw"]   = out["role_raw"]

    # seniority: reglas básicas por palabras clave en el título
    t = out["role_raw"].str.lower()
    out["seniority"] = None
    out.loc[t.str.contains(r"\b(junior|jr\.?)\b", regex=True, na=False), "seniority"] = "junior"
    out.loc[t.str.contains(r"\b(senior|sr\.?)\b", regex=True, na=False), "seniority"] = "senior"
    out.loc[t.str.contains(r"\b(lead|principal|head|director|vp|manager)\b", regex=True, na=False), "seniority"] = "lead"
    out.loc[out["seniority"].isna(), "seniority"] = "mid"

    # country: normalmente US (por estados US y formato Location)
    out["country"] = "US"

    # estado/región
    if "job_state" in df.columns:
        out["state_or_region"] = df["job_state"].astype(str).str.strip()
    else:
        out["state_or_region"] = None

    # remote_ratio: no existe -> NaN
    out["remote_ratio"] = pd.Series([pd.NA] * len(df), dtype="Float64")

    # company_size
    out["company_size"] = df["Size"].astype(str).str.strip()

    # employment_type: no existe -> Unknown
    out["employment_type"] = "Unknown"

    # salary_in_usd: avg_salary suele estar en "K" (72.0 = 72k)
    # Si tu avg_salary ya está en USD completos, quita el *1000. Lo comprobamos luego con stats.
    out["salary_in_usd"] = pd.to_numeric(df["avg_salary"], errors="coerce") * 1000

    return out[COMMON_COLS]


In [36]:
df1_n = normalize_df_ds_salaries(df1, "ds_salaries")
df2_n = normalize_df_ds_salaries(df2, "global_tech_salary")
df3_n = normalize_df_glassdoor(df3, "glassdoor")

all_salaries = pd.concat([df1_n, df2_n, df3_n], ignore_index=True)

print("df1_n:", df1_n.shape)
print("df2_n:", df2_n.shape)
print("df3_n:", df3_n.shape)
print("all_salaries:", all_salaries.shape)

display(all_salaries.head(10))


df1_n: (607, 12)
df2_n: (5000, 12)
df3_n: (742, 12)
all_salaries: (6349, 12)


C:\Users\Fiona A\AppData\Local\Temp\ipykernel_17552\4235837655.py:87: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  out.loc[t.str.contains(r"\b(junior|jr\.?)\b", regex=True, na=False), "seniority"] = "junior"
C:\Users\Fiona A\AppData\Local\Temp\ipykernel_17552\4235837655.py:88: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  out.loc[t.str.contains(r"\b(senior|sr\.?)\b", regex=True, na=False), "seniority"] = "senior"
C:\Users\Fiona A\AppData\Local\Temp\ipykernel_17552\4235837655.py:89: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  out.loc[t.str.contains(r"\b(lead|principal|head|director|vp|manager)\b", regex=True, na=False), "seniority"] = "lead"


,source_dataset,work_year,role_raw,role_label,seniority_raw,seniority,country,state_or_region,remote_ratio,company_size,employment_type,salary_in_usd
0,NaN,2020,Data Scientist,None,MI,mid,DE,None,0,L,FT,79833.0
1,NaN,2020,Machine Learning Scientist,None,SE,senior,JP,None,0,S,FT,260000.0
2,NaN,2020,Big Data Engineer,None,SE,senior,GB,None,50,M,FT,109024.0
3,NaN,2020,Product Data Analyst,None,MI,mid,HN,None,0,S,FT,20000.0
4,NaN,2020,Machine Learning Engineer,None,SE,senior,US,None,50,L,FT,150000.0
5,NaN,2020,Data Analyst,None,EN,junior,US,None,100,L,FT,72000.0
6,NaN,2020,Lead Data Scientist,None,SE,senior,US,None,100,S,FT,190000.0
7,NaN,2020,Data Scientist,None,MI,mid,HU,None,50,L,FT,35735.0
8,NaN,2020,Business Data Analyst,None,MI,mid,US,None,100,L,FT,135000.0
9,NaN,2020,Lead Data Engineer,None,SE,senior,NZ,None,50,S,FT,125000.0


In [37]:
def quality_report(df, name="df"):
    print(f"\n===== {name} quality report =====")
    print("rows:", len(df))
    print("\nNull%:")
    null_pct = (df.isna().mean() * 100).sort_values(ascending=False)
    print(null_pct.round(1))

    print("\nSalary stats (salary_in_usd):")
    print(df["salary_in_usd"].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]))

    print("\nTop countries:")
    print(df["country"].value_counts(dropna=False).head(10))

    print("\nTop company_size:")
    print(df["company_size"].value_counts(dropna=False).head(10))

quality_report(df1_n, "df1_n")
quality_report(df2_n, "df2_n")
quality_report(df3_n, "df3_n")
quality_report(all_salaries, "all_salaries")



===== df1_n quality report =====
rows: 607

Null%:
source_dataset     100.0
role_label         100.0
state_or_region    100.0
work_year            0.0
role_raw             0.0
seniority_raw        0.0
seniority            0.0
country              0.0
remote_ratio         0.0
company_size         0.0
employment_type      0.0
salary_in_usd        0.0
dtype: float64

Salary stats (salary_in_usd):
count       607.000000
mean     112297.869852
std       70957.259411
min        2859.000000
1%         5893.400000
5%        20000.000000
25%       62726.000000
50%      101570.000000
75%      150000.000000
95%      220110.000000
99%      403500.000000
max      600000.000000
Name: salary_in_usd, dtype: float64

Top countries:
US    332
GB     44
IN     30
CA     29
DE     25
FR     18
ES     15
GR     13
JP      7
PT      6
Name: country, dtype: int64

Top company_size:
M    326
L    198
S     83
Name: company_size, dtype: int64

===== df2_n quality report =====
rows: 5000

Null%:
source_dataset

In [38]:
COMMON_COLS = [
    "source_dataset",
    "work_year",
    "role_raw",
    "role_label",
    "seniority_raw",
    "seniority",
    "country",
    "state_or_region",
    "remote_ratio",
    "company_size",
    "employment_type",
    "salary_in_usd",
]

def map_experience_level_to_seniority(exp):
    if pd.isna(exp):
        return None
    exp = str(exp).strip().upper()
    mapping = {"EN": "junior", "MI": "mid", "SE": "senior", "EX": "lead"}
    return mapping.get(exp, None)

def normalize_df_ds_salaries(df, source_name: str):
    out = pd.DataFrame(index=df.index)  # ✅ clave

    out["source_dataset"]  = source_name
    out["work_year"]       = df["work_year"].astype("Int64")
    out["role_raw"]        = df["job_title"].astype(str).str.strip()

    out["role_label"]      = None  # lo crearemos después

    out["seniority_raw"]   = df["experience_level"].astype(str).str.strip()
    out["seniority"]       = df["experience_level"].apply(map_experience_level_to_seniority)

    out["country"]         = df["employee_residence"].astype(str).str.strip()

    out["state_or_region"] = None
    out["remote_ratio"]    = pd.to_numeric(df["remote_ratio"], errors="coerce")

    out["company_size"]    = df["company_size"].astype(str).str.strip()
    out["employment_type"] = df["employment_type"].astype(str).str.strip()

    out["salary_in_usd"]   = pd.to_numeric(df["salary_in_usd"], errors="coerce")

    return out[COMMON_COLS]

def normalize_df_glassdoor(df, source_name="glassdoor"):
    out = pd.DataFrame(index=df.index)  # ✅ clave

    out["source_dataset"]  = source_name
    out["work_year"]       = pd.Series([pd.NA] * len(df), index=df.index, dtype="Int64")

    out["role_raw"]        = df["Job Title"].astype(str).str.strip()
    out["role_label"]      = None

    out["seniority_raw"]   = out["role_raw"]

    t = out["role_raw"].str.lower()
    out["seniority"] = None
    # ✅ usar (?:...) para evitar warning de "match groups"
    out.loc[t.str.contains(r"\b(?:junior|jr\.?)\b", regex=True, na=False), "seniority"] = "junior"
    out.loc[t.str.contains(r"\b(?:senior|sr\.?)\b", regex=True, na=False), "seniority"] = "senior"
    out.loc[t.str.contains(r"\b(?:lead|principal|head|director|vp|manager)\b", regex=True, na=False), "seniority"] = "lead"
    out.loc[out["seniority"].isna(), "seniority"] = "mid"

    out["country"] = "US"

    out["state_or_region"] = df["job_state"].astype(str).str.strip() if "job_state" in df.columns else None

    out["remote_ratio"] = pd.Series([pd.NA] * len(df), index=df.index, dtype="Float64")

    out["company_size"] = df["Size"].astype(str).str.strip()
    out["employment_type"] = "Unknown"

    out["salary_in_usd"] = pd.to_numeric(df["avg_salary"], errors="coerce") * 1000

    return out[COMMON_COLS]


In [39]:
df1_n = normalize_df_ds_salaries(df1, "ds_salaries")
df2_n = normalize_df_ds_salaries(df2, "global_tech_salary")
df3_n = normalize_df_glassdoor(df3, "glassdoor")

all_salaries = pd.concat([df1_n, df2_n, df3_n], ignore_index=True)

print("df1_n:", df1_n.shape, "| source_dataset null%:", df1_n["source_dataset"].isna().mean())
print("df2_n:", df2_n.shape, "| source_dataset null%:", df2_n["source_dataset"].isna().mean())
print("df3_n:", df3_n.shape, "| source_dataset null%:", df3_n["source_dataset"].isna().mean())
print("all_salaries:", all_salaries.shape, "| source_dataset null%:", all_salaries["source_dataset"].isna().mean())

display(all_salaries.head())


df1_n: (607, 12) | source_dataset null%: 0.0
df2_n: (5000, 12) | source_dataset null%: 0.0
df3_n: (742, 12) | source_dataset null%: 0.0
all_salaries: (6349, 12) | source_dataset null%: 0.0


,source_dataset,work_year,role_raw,role_label,seniority_raw,seniority,country,state_or_region,remote_ratio,company_size,employment_type,salary_in_usd
0,ds_salaries,2020,Data Scientist,None,MI,mid,DE,None,0,L,FT,79833.0
1,ds_salaries,2020,Machine Learning Scientist,None,SE,senior,JP,None,0,S,FT,260000.0
2,ds_salaries,2020,Big Data Engineer,None,SE,senior,GB,None,50,M,FT,109024.0
3,ds_salaries,2020,Product Data Analyst,None,MI,mid,HN,None,0,S,FT,20000.0
4,ds_salaries,2020,Machine Learning Engineer,None,SE,senior,US,None,50,L,FT,150000.0


In [40]:
def normalize_company_size(x):
    if pd.isna(x):
        return "Unknown"
    s = str(x).strip()

    # ya viene como S/M/L
    if s in {"S","M","L"}:
        return s
    if s in {"-1", "Unknown", "nan", ""}:
        return "Unknown"

    # rangos tipo "51 to 200 employees"
    m = re.search(r"(\d+)\s*to\s*(\d+)\s*employees", s.lower())
    if m:
        lo = int(m.group(1)); hi = int(m.group(2))
        # bins
        if hi <= 200:
            return "S"
        elif hi <= 1000:
            return "M"
        else:
            return "L"

    # "10000+ employees"
    m2 = re.search(r"(\d+)\s*\+\s*employees", s.lower())
    if m2:
        n = int(m2.group(1))
        return "L" if n >= 1001 else "M"

    return "Unknown"

all_salaries["company_size_norm"] = all_salaries["company_size"].apply(normalize_company_size)

print(all_salaries["company_size"].value_counts(dropna=False).head(12))
print("\n--- normalized ---")
print(all_salaries["company_size_norm"].value_counts(dropna=False))


M                          4929
L                           533
1001 to 5000 employees      150
S                           145
501 to 1000 employees       134
10000+ employees            130
201 to 500 employees        117
51 to 200 employees          94
5001 to 10000 employees      76
1 to 50 employees            31
Unknown                       9
-1                            1
Name: company_size, dtype: int64

--- normalized ---
M          5180
L           889
S           270
Unknown      10
Name: company_size_norm, dtype: int64


In [41]:
import numpy as np

SALARY_ROLES = [
    "data_scientist",
    "data_engineer",
    "data_analyst",
    "machine_learning_engineer",
    "bi_engineer",
    "research_scientist",
    "project_manager",
    "other",
]

# Palabras que indican “scientist” no-tech (bio/med/lab) → las mandamos a other
NON_TECH_SCIENCE = r"\b(biology|biologist|immunolog|neuroscience|toxicolog|bacteriolog|pharma|clinical|lab|laboratory|hematolog|pathology|qc|viral|vector|assay|chemist|chemistry|molecule|cell|translational|biomarker|oncology|cancer|medic|hospital|healthcare|epidemiolog)\b"

def map_role_to_salary_taxonomy(title: str) -> str:
    if pd.isna(title):
        return "other"
    t = str(title).strip().lower()
    t = re.sub(r"\s+", " ", t)

    # 0) Project / Product Manager
    if re.search(r"\b(project manager|program manager)\b", t):
        return "project_manager"
    if re.search(r"\bdata (?:science|analytics) project manager\b", t):
        return "project_manager"
    if re.search(r"\bdata product manager\b", t):
        return "project_manager"

    # 1) Leadership / Management in data (asignación práctica)
    # Si contiene "machine learning" y es head/manager -> ML eng (más cercano a ML org)
    if re.search(r"\b(head|director|vp|manager|lead)\b", t) and re.search(r"\bmachine learning|ml|ai\b", t):
        return "machine_learning_engineer"

    # Si es "Head of Data" o "Data Lead" o "Data Manager" -> data_engineer (gestión de plataforma/infra común)
    if re.search(r"\b(head of data|data lead|data manager|data engineering manager)\b", t):
        return "data_engineer"
    # "Data Analytics Manager/Lead" tiende a analytics -> data_scientist (o data_analyst). Elijo DS por nivel.
    if re.search(r"\b(data analytics manager|data analytics lead|analytics manager)\b", t):
        return "data_scientist"

    # 2) MLOps / ML / AI / DL / NLP / CV
    if re.search(r"\b(mlops|mlo ps|machine learning ops|machine learning operations)\b", t):
        return "machine_learning_engineer"
    if re.search(r"\b(machine learning|ml|ai|deep learning|nlp|computer vision|cv)\b", t) and re.search(r"\b(engineer|developer|programmer|architect|ops|research engineer)\b", t):
        return "machine_learning_engineer"
    if re.search(r"\bmachine learning\b", t) and re.search(r"\bscientist\b", t):
        return "machine_learning_engineer"
    if re.search(r"\b(prompt engineer|ai developer|ai programmer|ai engineer|ai scientist)\b", t):
        return "machine_learning_engineer"

    # 3) BI roles
    if re.search(r"\b(bi|business intelligence)\b", t):
        # "Business Intelligence" pelado o manager -> bi_engineer (mejor que other)
        if re.search(r"\bmanager\b", t) or re.fullmatch(r"(?:bi|business intelligence)", t):
            return "bi_engineer"
        if re.search(r"\b(engineer|developer)\b", t):
            return "bi_engineer"
        if re.search(r"\banalyst\b", t):
            return "data_analyst"
        return "bi_engineer"

    if re.search(r"\b(analytics engineer)\b", t):
        return "data_engineer"

    # 4) Data Engineer / ETL / Pipeline / Infrastructure / Ops / Quality / Modeler
    if re.search(r"\b(data engineer|data infrastructure engineer|data pipeline engineer|data devops engineer|data ops engineer|data operations engineer)\b", t):
        return "data_engineer"
    if re.search(r"\b(etl developer|etl engineer|data integration engineer|data integration specialist|data integration developer)\b", t):
        return "data_engineer"
    if re.search(r"\b(big data)\b", t) and re.search(r"\b(engineer|developer|architect)\b", t):
        return "data_engineer"
    if re.search(r"\b(data architect|enterprise architect, data|aws data architect)\b", t):
        return "data_engineer"
    if re.search(r"\b(data modeler|data modeller|data developer)\b", t):
        return "data_engineer"
    if re.search(r"\b(data quality|data governance|data management|data operations|data ops)\b", t):
        return "data_engineer"

    # 5) Data Analyst family
    if re.search(r"\b(data analyst|analytics analyst|business data analyst|product data analyst|marketing data analyst|financial data analyst|sales data analyst|crm data analyst|risk data analyst|reporting analyst|insight analyst|data visualization specialist)\b", t):
        return "data_analyst"
    if re.search(r"\b(research analyst)\b", t):
        # suele ser analyst
        return "data_analyst"
    if re.search(r"\b(decision scientist)\b", t):
        return "data_scientist"

    # 6) Data Scientist
    if re.search(r"\bdata scientist\b", t) or re.search(r"\bdata science\b", t):
        return "data_scientist"
    if re.search(r"\b(applied scientist)\b", t):
        return "data_scientist"

    # 7) Research scientist family
    if re.search(r"\bscientist\b", t) and re.search(NON_TECH_SCIENCE, t):
        return "other"
    if re.search(r"\b(research scientist|research engineer|researcher)\b", t):
        if re.search(r"\b(machine learning|ml|ai|nlp|computer vision|deep learning)\b", t):
            return "machine_learning_engineer"
        return "research_scientist"

    return "other"



all_salaries["role_label_salary"] = all_salaries["role_raw"].apply(map_role_to_salary_taxonomy)

print(all_salaries["role_label_salary"].value_counts())
print(f"% other: {(all_salaries['role_label_salary'].eq('other').mean()*100):.2f}%")

print("\nTop 20 OTHER:")
print(all_salaries.loc[all_salaries["role_label_salary"]=="other", "role_raw"].value_counts().head(20))



data_scientist               1882
data_engineer                1825
data_analyst                 1089
machine_learning_engineer     886
research_scientist            271
other                         211
bi_engineer                   173
project_manager                12
Name: role_label_salary, dtype: int64
% other: 3.32%

Top 20 OTHER:
Data Specialist                                                                          22
Data Strategist                                                                           9
Robotics Software Engineer                                                                5
Cloud Database Engineer                                                                   5
R&D Specialist/ Food Scientist                                                            4
Data Analytics Consultant                                                                 4
Staff Scientist-Downstream Process Development                                            4
Food Scientist -

In [42]:
train_df = all_salaries[all_salaries["source_dataset"] != "glassdoor"].copy()

train_df["company_size"] = train_df["company_size_norm"]

FINAL_COLS_V1 = [
    "source_dataset",
    "work_year",
    "role_raw",
    "role_label_salary",
    "seniority_raw",
    "seniority",
    "country",
    "remote_ratio",
    "company_size",
    "employment_type",
    "salary_in_usd",
]

train_df_v1 = train_df[FINAL_COLS_V1].copy()

print("train_df_v1 shape:", train_df_v1.shape)
print("Null%:\n", (train_df_v1.isna().mean()*100).round(2))

out_path = Path("../3.processed/salaries_train_v3.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
train_df_v1.to_csv(out_path, index=False)
print("Saved:", out_path.resolve())


train_df_v1 shape: (5607, 11)
Null%:
 source_dataset       0.0
work_year            0.0
role_raw             0.0
role_label_salary    0.0
seniority_raw        0.0
seniority            0.0
country              0.0
remote_ratio         0.0
company_size         0.0
employment_type      0.0
salary_in_usd        0.0
dtype: float64
Saved: C:\Users\Fiona A\Desktop\IAPython\proyectos\TechCareer\TechCareer\backend\ml\data\model3_salaries\3.processed\salaries_train_v3.csv
